# Analisis Deskriptif Panel SIPRI–WDI untuk Artikel

Notebook ini dibuat sebagai alur analisis baru yang terpisah dari notebook tugas simulasi. Ia membaca `panel_regional_WDI_SIPRI_14negara_2010_2025.csv`, yaitu panel turunan dari ekspor SIPRI Military Expenditure Database dan respons API World Bank yang tersimpan lokal. Notebook belum menguji kausalitas atau kesiapan pertahanan.

**Penting:** `MS.MIL.MPRT.KD` berlabel *Arms imports (constant 1990 US$)*. Angka tersebut bukan TIV mentah atau harga kontrak. Belanja SIPRI memakai dolar konstan 2024, sehingga level kedua seri tidak dibandingkan nominal.


## 1. Pustaka dan jalur data

Sumber yang mendasari ekstraksi: SIPRI Military Expenditure Database (rilis revisi 27 April 2026) dan World Development Indicators. File panel memuat kode indikator, URL API WDI, DOI/rilis SIPRI, dan catatan negara pada baris terkait.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
PROJECT_ROOT = ROOT.parent if (ROOT.parent / "data").exists() else ROOT
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PANEL_PATH = DATA_PROCESSED_DIR / "panel_regional_WDI_SIPRI_14negara_2010_2025.csv" if (DATA_PROCESSED_DIR / "panel_regional_WDI_SIPRI_14negara_2010_2025.csv").exists() else ROOT / "panel_regional_WDI_SIPRI_14negara_2010_2025.csv"
EXP = "military_expenditure_constant_2024_usd_millions"
BURDEN = "military_expenditure_share_gdp_percent"
ARMS = "arms_imports_wdi_constant_1990_usd"

df = pd.read_csv(PANEL_PATH)
df["year"] = pd.to_numeric(df["year"], errors="raise").astype(int)
for col in [EXP, BURDEN, ARMS, "gdp_constant_2015_usd", "active_military_personnel_number"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
print(f"Panel: {len(df)} baris; {df['country'].nunique()} negara; tahun {df['year'].min()}–{df['year'].max()}")


## 2. Kelengkapan seri dan unit

Hitung missingness sebelum menentukan sampel untuk tiap indikator. Nilai kosong tetap kosong dan tidak diganti nol.


In [ ]:
fields = {
    "GDP konstan 2015 US$": "gdp_constant_2015_usd",
    "Belanja militer SIPRI konstan 2024 US$ juta": EXP,
    "Belanja militer SIPRI (% PDB)": BURDEN,
    "Personel angkatan bersenjata (WDI/IISS)": "active_military_personnel_number",
    "Impor senjata WDI (konstan 1990 US$)": ARMS,
}
coverage = pd.DataFrame([
    {"seri": label, "n_tersedia": int(df[col].notna().sum()), "n_total": len(df),
     "tahun_awal": int(df.loc[df[col].notna(), "year"].min()),
     "tahun_akhir": int(df.loc[df[col].notna(), "year"].max())}
    for label, col in fields.items()
])
coverage


## 3. Fungsi agregasi per jendela

Belanja militer diringkas sebagai rata-rata tahunan; burden sebagai rata-rata persentase tahunan; seri impor senjata WDI sebagai jumlah nilai lima tahun hanya jika semua lima observasi tersedia. Perubahan persentase dihitung dalam seri masing-masing.


In [ ]:
def compare_windows(data, start_a, end_a, start_b, end_b):
    a = data[data.year.between(start_a, end_a)]
    b = data[data.year.between(start_b, end_b)]
    def summarize(part, label):
        out = part.groupby("country").agg(
            expenditure_mean=(EXP, "mean"),
            burden_mean=(BURDEN, "mean"),
            arms_sum=(ARMS, lambda x: x.sum(min_count=(end_a-start_a+1 if label == "A" else end_b-start_b+1))),
            arms_n=(ARMS, "count"),
        ).reset_index()
        out = out.rename(columns={
            "expenditure_mean": f"expenditure_{label}",
            "burden_mean": f"burden_{label}",
            "arms_sum": f"arms_sum_{label}",
            "arms_n": f"arms_n_{label}",
        })
        return out
    left = summarize(a, "A")
    right = summarize(b, "B")
    out = left.merge(right, on="country", how="outer")
    out["expenditure_change_pct"] = (out.expenditure_B / out.expenditure_A - 1) * 100
    out["burden_change_pp"] = out.burden_B - out.burden_A
    complete = (out.arms_n_A == (end_a-start_a+1)) & (out.arms_n_B == (end_b-start_b+1))
    out["arms_imports_wdi_change_pct"] = pd.NA
    out.loc[complete, "arms_imports_wdi_change_pct"] = (out.loc[complete, "arms_sum_B"] / out.loc[complete, "arms_sum_A"] - 1) * 100
    return out

main = compare_windows(df, 2016, 2020, 2021, 2025)
sensitivity = compare_windows(df, 2015, 2019, 2020, 2024)


## 4. Hasil jendela utama dan sensitivitas

Kedua jendela saling bertumpang tindih, sehingga analisis sensitivitas bukan validasi independen.


In [ ]:
display(main.sort_values("country").round(3))
display(sensitivity.sort_values("country").round(3))


## 5. Arah perubahan lintas negara


In [ ]:
complete = main["arms_imports_wdi_change_pct"].notna()
exp_up = int((main.expenditure_change_pct > 0).sum())
arms_up = int((main.loc[complete, "arms_imports_wdi_change_pct"] > 0).sum())
divergent = int(((main.loc[complete, "expenditure_change_pct"] > 0) !=
                 (main.loc[complete, "arms_imports_wdi_change_pct"] > 0)).sum())
print(f"Belanja naik: {exp_up}/{len(main)} negara")
print(f"Indikator impor WDI naik: {arms_up}/{int(complete.sum())} negara lengkap")
print(f"Arah perubahan berbeda: {divergent}/{int(complete.sum())} negara lengkap")


## 6. Fokus Indonesia dan Australia


In [ ]:
main.loc[main.country.isin(["Indonesia", "Australia"]), [
    "country", "expenditure_A", "expenditure_B", "expenditure_change_pct",
    "burden_A", "burden_B", "burden_change_pp", "arms_sum_A", "arms_sum_B",
    "arms_imports_wdi_change_pct", "arms_n_A", "arms_n_B"
]].round(3)


## 7. Visualisasi perubahan di dalam masing-masing seri

Sumbu menunjukkan persentase perubahan setiap seri, bukan skala yang dapat dibandingkan. Plot tidak memasang model korelasi dan tidak menyatakan kausalitas.


In [ ]:
plot = main[main.arms_imports_wdi_change_pct.notna()].copy()
fig, ax = plt.subplots(figsize=(10, 6))
for _, row in plot.iterrows():
    color = "#d2643b" if row.country == "Indonesia" else "#137c8b" if row.country == "Australia" else "#667085"
    ax.scatter(row.expenditure_change_pct, row.arms_imports_wdi_change_pct, color=color, s=65)
    ax.annotate(row.country, (row.expenditure_change_pct, row.arms_imports_wdi_change_pct),
                xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.axhline(0, color="#475467", linestyle="--", linewidth=0.8)
ax.axvline(0, color="#475467", linestyle="--", linewidth=0.8)
ax.set_xlabel("Perubahan rata-rata belanja militer SIPRI, 2016–20 ke 2021–25 (%)")
ax.set_ylabel("Perubahan seri impor senjata WDI, jumlah 5 tahun (%)")
ax.set_title("Perubahan dua indikator pertahanan pada 12 negara dengan seri lengkap")
ax.grid(alpha=.2)
fig.tight_layout()


## 8. Interpretasi dan batas inferensi

- Belanja militer SIPRI adalah ukuran input sumber daya, bukan output kemampuan/kesiapan.
- Seri impor WDI `MS.MIL.MPRT.KD` berlabel dolar konstan 1990 dan bersumber dari SIPRI; jangan menyebut nilai ini TIV mentah, harga kontrak, atau pembayaran aktual tanpa memverifikasi transformasinya.
- Dolar konstan 2024 untuk belanja dan dolar konstan 1990 untuk indikator impor mempunyai basis harga/konsep yang berbeda. Bandingkan arah atau persentase perubahan di dalam masing-masing seri, bukan level nominal.
- Panel bersifat deskriptif, nonprobabilitas, 14 negara terpilih. Tidak ada uji signifikansi, prediksi, atau estimasi kausal.
- Naskah belum siap diajukan sampai pertanyaan riset dan kebaruan dibuktikan terhadap literatur dekat, serta seluruh angka diverifikasi pada rilis sumber yang diarsipkan.


## 9. Distribusi perubahan dan kesesuaian arah lintas negara

Ringkasan berikut menampilkan median, kuartil, rentang, dan tabulasi arah perubahan untuk jendela utama serta sensitivitas yang bertumpang tindih. Analisis tetap deskriptif; tidak ada uji signifikansi atau inferensi kausal.

In [ ]:
from pathlib import Path
import runpy
import pandas as pd

script_path = PROJECT_ROOT / "scripts" / "summarize_cross_country_changes.py"
if not script_path.exists():
    script_path = PROJECT_ROOT / "summarize_cross_country_changes.py"
runpy.run_path(str(script_path))

dist_path = DATA_PROCESSED_DIR / "ringkasan_distribusi_perubahan_negara_2026-10-24.csv"
if not dist_path.exists():
    dist_path = PROJECT_ROOT / "ringkasan_distribusi_perubahan_negara_2026-10-24.csv"

conc_path = DATA_PROCESSED_DIR / "tabulasi_kesesuaian_arah_belanja_impor_2026-10-24.csv"
if not conc_path.exists():
    conc_path = PROJECT_ROOT / "tabulasi_kesesuaian_arah_belanja_impor_2026-10-24.csv"

change_distribution = pd.read_csv(dist_path)
direction_concordance = pd.read_csv(conc_path)
display(change_distribution.round(3))
display(direction_concordance[direction_concordance.country_count > 0])
